# Finetuning Qwen3.5

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

from tqdm.auto import tqdm
from unsloth import FastVisionModel

from staged_qwen3_5_scivqa.config import (
    BASE_DIR,
    COMPETITION_DATA_DIR,
    DATA_DIR,
    ENABLE_THINKING,
    LORA_CHECKPOINT_FACTOID,
    MODEL_ID,
    PROMPT_FACTOID,
    TEMPERATURE,
    TOKEN_BUDGETS,
    TOP_K,
    TOP_P,
    MIN_P,
)
from staged_qwen3_5_scivqa.data import load_test_dataset


In [ ]:
MAX_NEW_TOKENS = TOKEN_BUDGETS["Factoid"]["max_new_tokens"]
MAX_SEQUENCE_LENGTH = TOKEN_BUDGETS["Factoid"]["max_sequence_length"]

LORA_CHECKPOINT = LORA_CHECKPOINT_FACTOID

DATA_DIR.mkdir(exist_ok=True)
CATEGORY = "test"

CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

STATE_FILE = DATA_DIR / f"submission_finetuning_{CATEGORY}__factoid_state.json"

SUMMARY_CACHE_PATH = DATA_DIR / f"submission_finetuning_summary_{CATEGORY}_state.json"
EXTRACTION_CACHE_PATH = (
    DATA_DIR / f"submission_finetuning_extraction_{CATEGORY}_state.json"
)

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen2-VL (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

Let's convert the dataset into the "correct" format for finetuning:

In [ ]:
summary_cache = None
with open(SUMMARY_CACHE_PATH) as f:
    summary_cache = json.load(f)

extraction_cache = None
with open(EXTRACTION_CACHE_PATH) as f:
    extraction_cache = json.load(f)

dataset = load_test_dataset(
    CATEGORY,
    summary_cache=summary_cache,
    extraction_cache=extraction_cache,
    answer_types=["Factoid"],
)

We look at how the conversations are structured for the first example:

In [ ]:
dataset[0]["messages"][0]["content"][1]["image"]

In [ ]:
dataset[0]

In [ ]:
question = "At what temperature were the ALD and ALE processes performed?"

for sample in dataset:
    if sample["meta"]["question"] == question:
        break

print(sample["meta"]["sub_fig"])

<a name="Submission"></a>
### 📜 Update the submission

Let's now update our submission! First, we must load the LoRA adapters we saved for inference!

In [ ]:
factoid_state = defaultdict(lambda: defaultdict(dict))

if STATE_FILE.exists():
    with open(STATE_FILE) as f:
        saved_factoid_state = json.load(f)

    # Reconstruct completed_tasks and factoid_state
    for s_id, sub_figs in saved_factoid_state.items():
        for s_fig, questions in sub_figs.items():
            # Handle the new dictionary format
            for q_text, ans_text in questions.items():
                task_id = f"{s_id}::{s_fig}::{q_text}"
                factoid_state[s_id][s_fig][q_text] = ans_text

    print(f"Loaded tracking state from {STATE_FILE}.")

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=LORA_CHECKPOINT,
    load_in_4bit=True,  # Set to False for 16bit LoRA
    max_seq_length=MAX_SEQUENCE_LENGTH,  # Must match the max_length used during training
)
FastVisionModel.for_inference(model)  # Enable for inference!

In [ ]:
pbar = tqdm(dataset, desc=f"Processing {CATEGORY} split", position=1, leave=False)
processed_cnt = 0
already_processed_cnt = 0

for data_item in pbar:
    meta = data_item["meta"]
    sample_id = meta["sample_id"]
    sub_fig = meta["sub_fig"]
    question_text = meta["question"]
    question_type = meta["question_type"]

    task_id = f"{sample_id}::{sub_fig}::{question_text}"

    # Check if already processed in a previous interrupted run
    if factoid_state.get(sample_id, {}).get(sub_fig, {}).get(question_text):
        already_processed_cnt += 1
        pbar.set_postfix(processed=processed_cnt, already=already_processed_cnt)
        continue

    pbar.set_description(f"Generating: {sample_id} | Sub: {sub_fig}")

    # Extract messages and image directly from the pre-built dataset item
    messages = data_item["messages"]
    image = messages[0]["content"][1]["image"]

    # Tokenize and format
    input_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=ENABLE_THINKING
    )
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        use_cache=True,
        temperature=TEMPERATURE,
        min_p=MIN_P,
        top_p=TOP_P,
        top_k=TOP_K,
    )

    # Decode only the newly generated tokens
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    ).strip()

    # 1. Update the Factoid-Specific Tracking State
    factoid_state[sample_id][sub_fig][question_text] = generated

    processed_cnt += 1
    pbar.set_postfix(processed=processed_cnt, already=already_processed_cnt)

    # 3. Persist State Every X Iterations (e.g., 5)
    if processed_cnt % 5 == 0:
        with open(STATE_FILE, "w") as f:
            json.dump(factoid_state, f, indent=4)

with open(STATE_FILE, "w") as f:
    json.dump(factoid_state, f, indent=4)

print(
    f"Pipeline execution complete! State fully synced.
"
    f"Summary -> Processed (Overwritten): {processed_cnt} | Already Completed: {already_processed_cnt}"
)